# Test NWS Orchestrator
This notebook allows you to load the complete Next-Word Suggestion pipeline and test it interactively.

In [1]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
current_dir = Path.cwd()
while current_dir.name and not (current_dir / "pyproject.toml").exists():
    current_dir = current_dir.parent
if str(current_dir) not in sys.path:
    sys.path.append(str(current_dir))

In [2]:
from src.services.nws.features.nwp.hybrid.model import HybridArabicPredictor
from src.services.nws.features.nwp.lstm.model import LSTMNWPModel
from src.services.nws.features.nwp.word_ngram.model import WordNGramLM
from src.services.nws.features.nwp.word_ngram.serializer import load_ngram_model
from src.services.nws.features.wac.char_ngram.model import CharNGramLM
from src.services.nws.features.wac.char_ngram.serializer import load_model
from src.services.nws.features.cache.manager import CacheManager
from src.services.nws.features.cache.idioms import IdiomsCache
from src.services.nws.features.cache.phrases import PhrasesCache
from src.services.nws.features.cache.user_lru import UserLRUCache
from src.services.nws.orchestrator import NWSOrchestrator
from src.core.schemas import Token
from src.services.nws.schemas import NWSInput

2026-06-26 03:21:04.597 | DEBUG    | src.services.ged.detectors.rule_based.registry:register_entry:100 - Registered entry: OT_ALIF_MAQSURA_ALA
2026-06-26 03:21:04.598 | DEBUG    | src.services.ged.detectors.rule_based.registry:register_entry:100 - Registered entry: OT_ALIF_MAQSURA_HATTA
2026-06-26 03:21:04.599 | DEBUG    | src.services.ged.detectors.rule_based.registry:register_entry:100 - Registered entry: OT_TANWIN_NASB_ON_ALIF
2026-06-26 03:21:04.599 | DEBUG    | src.services.ged.detectors.rule_based.registry:register_entry:100 - Registered entry: OT_IDGHAM_AN_MA
2026-06-26 03:21:04.599 | DEBUG    | src.services.ged.detectors.rule_based.registry:register_entry:100 - Registered entry: OT_IDGHAM_MIN_MA
2026-06-26 03:21:04.600 | DEBUG    | src.services.ged.detectors.rule_based.registry:register_entry:100 - Registered entry: OT_TA_MARBUTA_NOUN
2026-06-26 03:21:04.600 | DEBUG    | src.services.ged.detectors.rule_based.registry:register_entry:100 - Registered entry: OT_TA_MARBUTA_ADJ
2026

In [ ]:
print("Loading Models (This may take up to 45 seconds)...")
base_dir = current_dir / "src" / "services" / "nws" / "data"

# 1. Load NWP (Hybrid = LSTM + Kneser-Ney)
ngram_data = load_ngram_model(base_dir / "word_ngram_lm_lstm.msgpack.gz")
kn_model = WordNGramLM(ngram_data)
neural_model = LSTMNWPModel(
    model_path=str(base_dir / "best_model.pt"),
    sp_model_path=str(base_dir / "arabic_bpe.model"),
)
hybrid = HybridArabicPredictor(neural_model, kn_model)

# 2. Load WAC (Char N-Gram)
char_model = CharNGramLM(load_model(base_dir / "char_ngram_lm_lstm.msgpack.gz"))

# 3. Initialize Cache Manager
cache_manager = CacheManager(
    tier1=IdiomsCache(base_dir / "idioms.yaml"),
    tier2=PhrasesCache(base_dir / "phrases.yaml"),
    tier3=UserLRUCache(maxsize=1000)
)

# 4. Initialize Orchestrator
orchestrator = NWSOrchestrator(
    cache_manager=cache_manager,
    nwp_model=hybrid,
    wac_model=char_model,
)
print("Orchestrator is ready!")

Loading Models (This may take up to 45 seconds)...
Orchestrator is ready!


## Test: WAC Mode (Auto-Completion)
Tests auto-completing a partially typed word.

In [4]:
input_wac = NWSInput(
    tokens=[Token(form="ذهب"), Token(form="الطلاب"), Token(form="إلى")],
    morph_features=[],
    current_fragment="المدر",
    mode="WAC",
    top_k=5
)

output_wac = orchestrator.predict(input_wac, debug=True)

print("\n--- Suggestions ---")
for s in output_wac.suggestions:
    print(f"{s.rank + 1}. {s.word} (Confidence: {s.score:.2%}) [{s.source}]")

ذهب الطلاب إلى 
[DEBUG NWS] Cache MISS - Evaluated using WAC model
[DEBUG NWS]   -> 0: المدرسه (score: 0.3185)
[DEBUG NWS]   -> 1: المدرسيه (score: 0.1857)
[DEBUG NWS]   -> 2: المدرجات (score: 0.1759)
[DEBUG NWS]   -> 3: المدرب (score: 0.1674)
[DEBUG NWS]   -> 4: المدرجه (score: 0.1526)

--- Suggestions ---
1. المدرسه (Confidence: 31.85%) [model]
2. المدرسيه (Confidence: 18.57%) [model]
3. المدرجات (Confidence: 17.59%) [model]
4. المدرب (Confidence: 16.74%) [model]
5. المدرجه (Confidence: 15.26%) [model]


## Test: NWP Mode (Next Word Prediction)
Tests predicting the next entire word.

In [5]:
input_nwp = NWSInput(
    tokens=[Token(form="ذهب"), Token(form="الطلاب"), Token(form="إلى")],
    morph_features=[],
    current_fragment=None,
    mode="NWP",
    top_k=5
)

output_nwp = orchestrator.predict(input_nwp, debug=True)

print("\n--- Suggestions ---")
for s in output_nwp.suggestions:
    print(f"{s.rank + 1}. {s.word} (Confidence: {s.score:.2%}) [{s.source}]")

[DEBUG NWS] Cache MISS - Evaluated using NWP model
[DEBUG NWS]   -> 0: في (score: 0.5599)
[DEBUG NWS]   -> 1: ان (score: 0.2093)
[DEBUG NWS]   -> 2: و (score: 0.1455)
[DEBUG NWS]   -> 3: ا (score: 0.0588)
[DEBUG NWS]   -> 4: م (score: 0.0265)

--- Suggestions ---
1. في (Confidence: 55.99%) [model]
2. ان (Confidence: 20.93%) [model]
3. و (Confidence: 14.55%) [model]
4. ا (Confidence: 5.88%) [model]
5. م (Confidence: 2.65%) [model]
